# RAG for GoF Design Pattern Classification from PlantUML

A reproducible Vector RAG + LLM baseline.

**Important:** the corpus annotations (`Design pattern ...` and `DP role: ...`) are removed from query text to prevent label leakage, but retained in indexed training examples as labeled demonstrations.


## 1. Install dependencies

In [7]:
print("Corpus path:", CORPUS_DIR)
print("Exists:", CORPUS_DIR.exists())
print("Is directory:", CORPUS_DIR.is_dir())

files = list(CORPUS_DIR.glob("*.puml"))

print("Number of PUML files:", len(files))

for f in files[:10]:
    print(f)

Corpus path: /PMART_UML/knowledge_base
Exists: False
Is directory: False
Number of PUML files: 0


In [ ]:
%pip install -q sentence-transformers faiss-cpu scikit-learn pandas matplotlib openai tqdm

## 2. Configuration

Put your `.puml` files under `CORPUS_DIR`.

Set your API key outside the notebook:

```bash
export OPENAI_API_KEY="..."
```


In [15]:
from pathlib import Path
import os, re, json, random, hashlib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import faiss
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt

CORPUS_DIR = Path("/P-MART-RAG/knowledge_base")  # CHANGE THIS
OUTPUT_DIR = Path("outputs/rag_design_patterns")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 5
RANDOM_SEED = 42
LLM_MODEL = os.getenv("RAG_LLM_MODEL", "gpt-5.6-mini")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(CORPUS_DIR.resolve())





/P-MART-RAG/knowledge_base


## 3. Load the corpus

In [16]:
def load_corpus(corpus_dir):
    files = sorted(corpus_dir.rglob("*.puml"))
    if not files:
        raise FileNotFoundError(f"No .puml files found under {corpus_dir.resolve()}")
    records = []
    for path in files:
        text = path.read_text(encoding="utf-8", errors="replace")
        records.append({
            "id": hashlib.sha1(str(path).encode()).hexdigest()[:12],
            "filename": str(path),
            "plantuml": text,
        })
    return pd.DataFrame(records)

df = load_corpus(CORPUS_DIR)
print(f"Loaded {len(df)} diagrams")
display(df.head(3)[["id", "filename"]])


FileNotFoundError: No .puml files found under /P-MART-RAG/knowledge_base

## 4. Extract pattern and DP-role annotations

In [ ]:
PATTERN_RE = re.compile(r"^\s*'\s*Design\s+pattern\s*:?\s*(.+?)\s*$", re.I | re.M)
ROLE_RE = re.compile(r"^\s*'\s*DP\s+role\s*:\s*(.+?)\s*$", re.I | re.M)

def extract_annotations(text):
    patterns = PATTERN_RE.findall(text)
    roles = ROLE_RE.findall(text)
    return (patterns[0].strip() if patterns else None, [r.strip() for r in roles])

parsed = df["plantuml"].apply(extract_annotations)
df["pattern"] = parsed.apply(lambda x: x[0])
df["roles"] = parsed.apply(lambda x: x[1])

print("Labeled:", df["pattern"].notna().sum())
print("Unlabeled:", df["pattern"].isna().sum())
display(df[["filename", "pattern", "roles"]].head(10))
display(df["pattern"].value_counts(dropna=False))


## 5. Normalize PlantUML

In [ ]:
def remove_annotations(text):
    lines = []
    for line in text.splitlines():
        if re.match(r"^\s*'\s*Design\s+pattern\s*:", line, re.I):
            continue
        if re.match(r"^\s*'\s*DP\s+role\s*:", line, re.I):
            continue
        lines.append(line)
    return "\n".join(lines)

def remove_doc_comments(text):
    output = []
    in_block = False
    for line in text.splitlines():
        s = line.strip()
        if s.startswith("'/**") or s.startswith("'/*"):
            in_block = True
            continue
        if in_block:
            if s.endswith("*/") or s.startswith("' */"):
                in_block = False
            continue
        if s.startswith("'"):
            continue
        output.append(line)
    return "\n".join(output)

def normalize_plantuml(text):
    text = remove_annotations(text)
    text = remove_doc_comments(text)
    return "\n".join(line.rstrip() for line in text.splitlines() if line.strip())

df["normalized_plantuml"] = df["plantuml"].apply(normalize_plantuml)


## 6. Build indexed-example and query documents

In [ ]:
def build_index_document(row):
    roles = ", ".join(row["roles"]) if row["roles"] else "not annotated"
    return (
        "GoF design pattern example.\n"
        f"Known pattern: {row['pattern']}\n"
        f"Annotated roles: {roles}\n\n"
        "UML / PlantUML structure:\n"
        f"{row['normalized_plantuml']}"
    )

def build_query_document(row):
    return (
        "UML / PlantUML design to classify.\n"
        "The design-pattern label is intentionally omitted.\n\n"
        "Structural representation:\n"
        f"{row['normalized_plantuml']}"
    )

df["index_document"] = df.apply(build_index_document, axis=1)
df["query_document"] = df.apply(build_query_document, axis=1)
display(df[["pattern", "roles", "index_document"]].head(1))


## 7. Filter and inspect class balance

In [ ]:
df_labeled = df[df["pattern"].notna()].copy()
df_labeled["pattern"] = df_labeled["pattern"].str.strip()
print("Examples:", len(df_labeled))
print("Patterns:", df_labeled["pattern"].nunique())
display(df_labeled["pattern"].value_counts().rename_axis("pattern").reset_index(name="count"))


## 8. Train/test split

This prototype uses a stratified split. For the paper, prefer a project/family-level split if multiple diagrams originate from the same source system.


In [ ]:
train_df, test_df = train_test_split(
    df_labeled, test_size=0.20, random_state=RANDOM_SEED, stratify=df_labeled["pattern"]
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print("Train:", len(train_df), "Test:", len(test_df))


## 9. Generate embeddings

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

train_embeddings = embedding_model.encode(
    train_df["index_document"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
dimension = train_embeddings.shape[1]
print("Embedding dimension:", dimension)


## 10. Build FAISS index

In [ ]:
index = faiss.IndexFlatIP(dimension)
index.add(train_embeddings.astype("float32"))
print("FAISS vectors:", index.ntotal)


## 11. Retriever

In [ ]:
def retrieve(query_row, top_k=TOP_K):
    query_text = build_query_document(query_row)
    query_embedding = embedding_model.encode(
        [query_text], convert_to_numpy=True, normalize_embeddings=True
    )
    scores, indices = index.search(query_embedding.astype("float32"), top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        row = train_df.iloc[int(idx)]
        results.append({
            "id": row["id"],
            "filename": row["filename"],
            "pattern": row["pattern"],
            "roles": row["roles"],
            "score": float(score),
            "plantuml": row["normalized_plantuml"],
        })
    return results

display(pd.DataFrame([
    {"pattern": r["pattern"], "score": r["score"]}
    for r in retrieve(test_df.iloc[0], TOP_K)
]))


## 12. Retrieval Recall@k

In [ ]:
def retrieval_recall_at_k(eval_df, k):
    hits = []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
        hits.append(row["pattern"] in {r["pattern"] for r in retrieve(row, k)})
    return float(np.mean(hits))

for k in [1, 3, 5, 10]:
    print(f"Recall@{k}: {retrieval_recall_at_k(test_df, k):.4f}")


## 13. Inspect retrieved context

In [ ]:
def show_retrieval(row, top_k=5):
    print("=" * 80)
    print("QUERY")
    print("=" * 80)
    print(row["query_document"])
    print("\n" + "=" * 80)
    print("RETRIEVED EXAMPLES")
    print("=" * 80)
    for i, r in enumerate(retrieve(row, top_k), 1):
        print(f"\n--- Example {i} ---")
        print("Pattern:", r["pattern"])
        print("Similarity:", round(r["score"], 4))
        print("Roles:", r["roles"])
        print(r["plantuml"][:3000])

show_retrieval(test_df.iloc[0], TOP_K)


## 14. LLM classifier

The LLM receives the query plus retrieved labeled examples and returns JSON.


In [ ]:
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY is not set.")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
ALLOWED_PATTERNS = sorted(train_df["pattern"].unique().tolist())

def build_llm_prompt(query_row, retrieved):
    blocks = []
    for i, r in enumerate(retrieved, 1):
        roles = ", ".join(r["roles"]) if r["roles"] else "not available"
        blocks.append(
            "RETRIEVED EXAMPLE " + str(i) + "\n"
            + "Pattern: " + r["pattern"] + "\n"
            + "Roles: " + roles + "\n"
            + f"Similarity: {r['score']:.4f}\n\n"
            + "PlantUML:\n" + r["plantuml"]
        )

    examples_text = "\n\n".join(blocks)

    prompt = (
        "You are an expert in Model-Driven Engineering, UML, and Gang-of-Four design patterns.\n\n"
        "Classify the query diagram into exactly one of these patterns:\n"
        + json.dumps(ALLOWED_PATTERNS, ensure_ascii=False)
        + "\n\n"
        "Do not simply choose the highest-similarity example. "
        "Use structural evidence such as inheritance, implementation, association, "
        "aggregation/composition, dependency direction, responsibilities, and collaboration structure.\n\n"
        "QUERY DIAGRAM\n"
        + query_row["normalized_plantuml"]
        + "\n\nRETRIEVED LABELED EXAMPLES\n"
        + examples_text
        + "\n\nReturn ONLY valid JSON with this schema:\n"
        + '{"prediction":"<one allowed pattern>","confidence":0.0,'
          '"evidence":["..."],"alternative":"<pattern>",'
          '"alternative_reason":"...","retrieved_example_ids":["..."]}'
    )
    return prompt

def classify_with_llm(query_row, top_k=TOP_K, model=LLM_MODEL):
    retrieved = retrieve(query_row, top_k)
    prompt = build_llm_prompt(query_row, retrieved)

    response = client.chat.completions.create(
        model=model,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a precise UML design-pattern classifier. Return only JSON."},
            {"role": "user", "content": prompt},
        ],
    )

    prediction = json.loads(response.choices[0].message.content)
    return {"prediction": prediction, "retrieved": retrieved, "prompt": prompt}


## 15. Test one classification

In [ ]:
if os.getenv("OPENAI_API_KEY"):
    result = classify_with_llm(test_df.iloc[0], TOP_K)
    print(json.dumps(result["prediction"], indent=2, ensure_ascii=False))
else:
    print("Set OPENAI_API_KEY first.")


## 16. Run the full experiment

Use `MAX_TEST_EXAMPLES = 10` while debugging. Set it to `None` for the full test set.


In [ ]:
MAX_TEST_EXAMPLES = 10

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set.")

eval_df = test_df.copy()
if MAX_TEST_EXAMPLES is not None:
    eval_df = eval_df.head(MAX_TEST_EXAMPLES).copy()

results = []

for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    try:
        output = classify_with_llm(row, TOP_K)
        pred = output["prediction"]
        results.append({
            "id": row["id"],
            "filename": row["filename"],
            "gold_pattern": row["pattern"],
            "predicted_pattern": pred.get("prediction"),
            "confidence": pred.get("confidence"),
            "evidence": json.dumps(pred.get("evidence", []), ensure_ascii=False),
            "alternative": pred.get("alternative"),
            "alternative_reason": pred.get("alternative_reason"),
            "retrieved_example_ids": json.dumps(pred.get("retrieved_example_ids", []), ensure_ascii=False),
            "retrieved_patterns": json.dumps([r["pattern"] for r in output["retrieved"]], ensure_ascii=False),
            "retrieval_scores": json.dumps([r["score"] for r in output["retrieved"]]),
        })
    except Exception as exc:
        results.append({
            "id": row["id"],
            "filename": row["filename"],
            "gold_pattern": row["pattern"],
            "predicted_pattern": None,
            "confidence": None,
            "evidence": "[]",
            "alternative": None,
            "alternative_reason": str(exc),
            "retrieved_example_ids": "[]",
            "retrieved_patterns": "[]",
            "retrieval_scores": "[]",
        })

results_df = pd.DataFrame(results)
results_path = OUTPUT_DIR / "rag_predictions.csv"
results_df.to_csv(results_path, index=False)
print("Saved:", results_path)
display(results_df.head())


## 17. Metrics

In [ ]:
valid = results_df.dropna(subset=["predicted_pattern"])
y_true = valid["gold_pattern"]
y_pred = valid["predicted_pattern"]

accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro", zero_division=0
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro-P  : {precision:.4f}")
print(f"Macro-R  : {recall:.4f}")
print(f"Macro-F1 : {f1:.4f}")
print(classification_report(y_true, y_pred, zero_division=0))


## 18. Confusion matrix

In [ ]:
labels = sorted(set(y_true) | set(y_pred))
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.55), max(7, len(labels) * 0.55)))
ax.imshow(cm)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90)
ax.set_yticklabels(labels)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("RAG + LLM GoF Pattern Classification")

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
cm_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", cm_path)


## 19. Per-pattern performance

In [ ]:
report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
per_pattern = pd.DataFrame(report).T.reset_index().rename(columns={"index": "pattern"})
display(per_pattern)


## 20. Retrieval versus classification

In [ ]:
analysis = results_df.copy()
analysis["retrieved_patterns_list"] = analysis["retrieved_patterns"].apply(json.loads)
analysis["retrieval_hit"] = analysis.apply(
    lambda r: r["gold_pattern"] in r["retrieved_patterns_list"], axis=1
)

display(
    analysis.groupby("gold_pattern")["retrieval_hit"]
    .agg(["mean", "count"])
    .reset_index()
    .rename(columns={"mean": "Recall@K"})
)


# 21. Research roadmap

This notebook is the **Vector RAG baseline**.

Next compare:

1. LLM-only
2. Raw PlantUML RAG
3. Structural RAG
4. GraphRAG
5. Hybrid Vector + GraphRAG

Use the same held-out test set for all systems. For the final paper, prefer project/family-level splitting to prevent near-duplicate leakage.
